In [1]:
# 1. Force Jupyter to install the Delta library and its Java files locally
!pip install delta-spark==3.1.0

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 2. Build the session WITHOUT the spark.jars.packages line
builder = (
    SparkSession.builder
    .appName("Z5008-Assign2-Musa")
    .master("local[*]")
    # --- ADD THESE TWO LINES TO FIX THE ERROR ---
    .config("spark.eventLog.enabled", "false")
    .config("spark.ui.enabled", "true") 
    # --------------------------------------------
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "admin")
    .config("spark.hadoop.fs.s3a.secret.key", "bigdata123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.driver.memory", "2g") 
)

spark = configure_spark_with_delta_pip(
    builder, 
    extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4", "com.amazonaws:aws-java-sdk-bundle:1.12.262"]
).getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# --- TASK 1c VERIFICATION OUTPUT ---
print("--- Spark Verification ---")
print(f"(i)   Spark Version: {spark.version}")
print(f"(ii)  Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"(iii) Spark Master: {spark.sparkContext.master}")

--- Spark Verification ---
(i)   Spark Version: 3.5.0
(ii)  Default Parallelism: 4
(iii) Spark Master: local[*]


In [2]:
# --- Task 2a: Create Dataset (10,000 rows) ---
import pandas as pd
import random
from pyspark.sql import functions as F
from pyspark.sql.types import *

random.seed(42)
n = 10000
pdf = pd.DataFrame({
    "txn_id":      range(1, n + 1),
    "customer_id": [f"C{i % 500 + 1:04d}" for i in range(n)],
    "merchant":    ["ShopRite", "Nakumatt", "Jumia", "M-Pesa", "KCB Bank"] * 2000,
    "city":        ["Nairobi", "Dar es Salaam", "Zanzibar", "Kampala", "Mombasa"] * 2000,
    "category":    ["groceries", "electronics", "finance", "transport", "dining"] * 2000,
    "amount_usd":  [round(abs(random.gauss(150, 120)), 2) for _ in range(n)],
    "currency":    ["KES", "TZS", "TZS", "KES", "KES"] * 2000,
    "txn_date":    pd.date_range("2026-01-01", periods=n, freq="h"),
    "is_fraud":    [False] * 9800 + [True] * 200
})

txn_schema = StructType([
    StructField("txn_id", IntegerType(), False),
    StructField("customer_id", StringType(), True),
    StructField("merchant", StringType(), True),
    StructField("city", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount_usd", DoubleType(), True),
    StructField("currency", StringType(), True),
    StructField("txn_date", TimestampType(), True),
    StructField("is_fraud", BooleanType(), True)
])

df = spark.createDataFrame(pdf, schema=txn_schema)
TABLE_PATH = "s3a://warehouse/bronze/transactions"

# Write to Delta partitioned by 'city' (Version 0)
df.write.format("delta").mode("overwrite").partitionBy("city").save(TABLE_PATH)
print(f"Task 2a: Written {df.count()} rows to {TABLE_PATH}")

Task 2a: Written 10000 rows to s3a://warehouse/bronze/transactions


In [3]:
# --- Task 2b: Three Meaningful SQL Queries ---
df_delta = spark.read.format("delta").load(TABLE_PATH)
df_delta.createOrReplaceTempView("transactions")

print("Query 1: Total Spend and Fraud Count by City")
spark.sql("""
    SELECT city, COUNT(*) as total_txns, SUM(amount_usd) as total_revenue, SUM(CAST(is_fraud AS INT)) as fraud_cases
    FROM transactions GROUP BY city ORDER BY total_revenue DESC
""").show()

print("Query 2: Average Transaction Amount per Category")
spark.sql("""
    SELECT category, AVG(amount_usd) as avg_spend, MAX(amount_usd) as max_spend
    FROM transactions GROUP BY category
""").show()

print("Query 3: Merchant Performance (Count and Sum)")
spark.sql("""
    SELECT merchant, COUNT(txn_id) as txn_count, ROUND(SUM(amount_usd), 2) as total_val
    FROM transactions GROUP BY merchant HAVING txn_count > 100
""").show()

Query 1: Total Spend and Fraud Count by City
+-------------+----------+------------------+-----------+
|         city|total_txns|     total_revenue|fraud_cases|
+-------------+----------+------------------+-----------+
|      Mombasa|      2000|324724.84000000014|         40|
|     Zanzibar|      2000| 323782.4800000001|         40|
|      Nairobi|      2000| 321350.3299999999|         40|
|Dar es Salaam|      2000|         321307.77|         40|
|      Kampala|      2000|318224.18000000005|         40|
+-------------+----------+------------------+-----------+

Query 2: Average Transaction Amount per Category
+-----------+------------------+---------+
|   category|         avg_spend|max_spend|
+-----------+------------------+---------+
|     dining|162.36242000000007|   600.25|
|  transport|159.11209000000002|   527.88|
|electronics|        160.653885|   587.55|
|    finance|161.89124000000004|   580.66|
|  groceries|160.67516499999994|   501.54|
+-----------+------------------+-------

In [4]:
# --- Task 3a: Create 3 Versions ---
# Version 1: Append 500 rows
append_pdf = pdf.iloc[:500].copy()
append_pdf['txn_id'] = append_pdf['txn_id'] + 10000
df_v1 = spark.createDataFrame(append_pdf, schema=txn_schema)
df_v1.write.format("delta").mode("append").save(TABLE_PATH)

# Version 2: OPTIMIZE (this creates a new version in history)
spark.sql(f"OPTIMIZE delta.`{TABLE_PATH}`")

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,numBatches:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,preserveInsertionOrder:boolean,numFilesSkippedToReduceWriteAmplification:bigint,numBytesSkippedToReduceWriteAmplification:bigint,startTimeMs:bigint,endTimeMs:bigint,totalClusterParallelism:bigint,totalScheduledTasks:bigint,autoCompactParallelismStats:struct<maxClusterActiveParallelism:bigint,minClusterActiveParallelism:bigint,maxSessionActiveParallelism:bigint,minSessionActiveParallelism:bigint>,de

In [5]:
# --- Task 3b: Describe History ---
print("Task 3b: Table History")
spark.sql(f"DESCRIBE HISTORY delta.`{TABLE_PATH}`").select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)


Task 3b: Table History
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                             |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|19     |2026-03-22 13:57:29|OPTIMIZE |{numRemovedFiles -> 40, numRemovedBytes -> 267092, p25FileSize -> 33397, numDeletionVectorsRemoved -> 0,

In [6]:
# --- Task 3c: Compare Version 0 and Version 2 ---
v0 = spark.read.format("delta").option("versionAsOf", 0).load(TABLE_PATH)
v2 = spark.read.format("delta").option("versionAsOf", 2).load(TABLE_PATH)
print(f"Version 0: {v0.count()} rows, total = {v0.select(F.sum('amount_usd')).collect()[0][0]}")
print(f"Version 2: {v2.count()} rows, total = {v2.select(F.sum('amount_usd')).collect()[0][0]}")


Version 0: 10000 rows, total = 1609389.6000000008
Version 2: 10000 rows, total = 1609389.6000000008


In [7]:
# --- Task 3d: Restore to Version 0 ---
spark.sql(f"RESTORE TABLE delta.`{TABLE_PATH}` TO VERSION AS OF 0")
print(f"Restored! Current row count: {spark.read.format('delta').load(TABLE_PATH).count()}")

Restored! Current row count: 10000


In [8]:
# --- Task 4a: Create Silver Table ---
SILVER_PATH = "s3a://warehouse/silver/customer_summary"
spark.sql(f"""
    SELECT customer_id, SUM(amount_usd) as total_spend, COUNT(*) as txn_count, MAX(txn_date) as last_active
    FROM delta.`{TABLE_PATH}` GROUP BY customer_id
""").write.format("delta").mode("overwrite").save(SILVER_PATH)


In [9]:
# --- Task 4b & 4c: Prepare Batch and Run MERGE ---
from delta.tables import DeltaTable

# Create updates: 10 updates, 5 new inserts, 3 deletes (flagged)
merge_data = []
for i in range(1, 11): merge_data.append((f"C{i:04d}", 9999.99, 100, "active")) # Updates
for i in range(600, 605): merge_data.append((f"C{i:04d}", 500.00, 1, "active")) # Inserts
for i in range(11, 14): merge_data.append((f"C{i:04d}", 0.0, 0, "deleted"))      # Deletes

updates_df = spark.createDataFrame(merge_data, ["customer_id", "total_spend", "txn_count", "status"])

target_table = DeltaTable.forPath(spark, SILVER_PATH)
target_table.alias("t").merge(
    updates_df.alias("s"), "t.customer_id = s.customer_id"
).whenMatchedDelete(condition="s.status = 'deleted'"
).whenMatchedUpdate(set={"total_spend": "s.total_spend", "txn_count": "s.txn_count"}
).whenNotMatchedInsert(values={"customer_id": "s.customer_id", "total_spend": "s.total_spend", "txn_count": "s.txn_count", "last_active": "current_timestamp()"}
).execute()

In [10]:
# --- Task 4d: Verify ---
print("Verification:")
spark.read.format("delta").load(SILVER_PATH).filter("customer_id IN ('C0001', 'C0011', 'C0601')").show()

Verification:
+-----------+-----------+---------+--------------------+
|customer_id|total_spend|txn_count|         last_active|
+-----------+-----------+---------+--------------------+
|      C0001|    9999.99|      100| 2027-01-31 20:00:00|
|      C0601|      500.0|        1|2026-03-22 13:58:...|
+-----------+-----------+---------+--------------------+



In [11]:
# --- Task 5a: Schema Enforcement ---
try:
    extra_col_df = df.limit(1).withColumn("new_extra_field", F.lit("error"))
    extra_col_df.write.format("delta").mode("append").save(TABLE_PATH)
except Exception as e:
    print("Schema Enforcement Error caught successfully:")
    print(str(e)[:200])



Schema Enforcement Error caught successfully:
A schema mismatch detected when writing to the Delta table (Table ID: 22c55468-f503-4d07-b572-72572b2e7273).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option(


In [12]:
# --- Task 5b: Schema Evolution ---
df.limit(5).withColumn("discount_code", F.lit("SAVE10")) \
  .write.format("delta").mode("append").option("mergeSchema", "true").save(TABLE_PATH)
spark.read.format("delta").load(TABLE_PATH).select("txn_id", "discount_code").show(10)



+------+-------------+
|txn_id|discount_code|
+------+-------------+
|  5122|         NULL|
|  5127|         NULL|
|  5132|         NULL|
|  5137|         NULL|
|  5142|         NULL|
|  5147|         NULL|
|  5152|         NULL|
|  5157|         NULL|
|  5162|         NULL|
|  5167|         NULL|
+------+-------------+
only showing top 10 rows



In [13]:
# --- Task 5c: OPTIMIZE ---
# Capture screenshot of MinIO before running this
spark.sql(f"OPTIMIZE delta.`{TABLE_PATH}` ZORDER BY (customer_id)")
print("Table Optimized with Z-Order on customer_id")

Table Optimized with Z-Order on customer_id
